In [10]:
import pandas as pd
import numpy as np

In [11]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Burari Crossing, Delhi - IMD.xlsx",skiprows=16)

In [15]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,138.33,216.01,19.66,11.68,22.20,0.680,22.55,0
1,02-01-2025 00:00,03-01-2025 00:00,172.79,262.59,16.59,11.69,19.71,0.750,24.61,0
2,03-01-2025 00:00,04-01-2025 00:00,58.43,362.23,13.86,14.03,27.50,1.400,19.11,0
3,04-01-2025 00:00,05-01-2025 00:00,58.43,326.44,13.86,17.47,36.81,0.925,19.69,0
4,05-01-2025 00:00,06-01-2025 00:00,153.82,234.52,16.86,13.04,20.64,1.000,27.74,0
...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,58.43,148.71,13.86,20.55,28.54,0.925,55.71,0
316,13-11-2025 00:00,14-11-2025 00:00,58.43,148.71,13.86,20.55,21.32,0.925,53.48,0
317,14-11-2025 00:00,15-11-2025 00:00,58.43,369.64,13.86,20.55,12.64,1.470,50.02,0
318,15-11-2025 00:00,16-11-2025 00:00,58.43,148.71,13.86,20.55,19.19,1.660,47.41,0


In [12]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 11)


In [13]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Benzene']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
TOT-RF       0
dtype: int64


In [16]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [17]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')

In [18]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 10)
   From Date    To Date   PM2.5    PM10     NO    NO2    NOx     CO  Ozone  \
0 2025-01-01 2025-02-01  138.33  216.01  13.86  11.68  22.20  0.680  22.55   
1 2025-02-01 2025-03-01   58.43  262.59  13.86  11.69  19.71  0.750  24.61   
2 2025-03-01 2025-04-01   58.43  362.23  13.86  14.03  27.50  1.400  19.11   
3 2025-04-01 2025-05-01   58.43  326.44  13.86  17.47  36.81  0.925  19.69   
4 2025-05-01 2025-06-01   58.43  234.52  13.86  13.04  20.64  1.000  27.74   

   TOT-RF  
0       0  
1       0  
2       0  
3       0  
4       0  


In [19]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [20]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,TOT-RF
0,2025-01-01,2025-02-01,3.159561,0.765647,0.164216,-0.971723,0.102375,-1.032027,-1.037026,0.0
1,2025-02-01,2025-03-01,0.061955,1.326799,0.164216,-0.970989,-0.135695,-0.717697,-0.885631,0.0
2,2025-03-01,2025-04-01,0.061955,2.527167,0.164216,-0.799163,0.609111,2.201081,-1.289842,0.0
3,2025-04-01,2025-05-01,0.061955,2.096003,0.164216,-0.546563,1.499245,0.068128,-1.247216,0.0
4,2025-05-01,2025-06-01,0.061955,0.988638,0.164216,-0.871858,-0.046777,0.404910,-0.655598,0.0
...,...,...,...,...,...,...,...,...,...,...
315,2025-12-11,NaT,0.061955,-0.045120,0.164216,-0.320398,0.708546,0.068128,1.399999,0.0
316,NaT,NaT,0.061955,-0.045120,0.164216,-0.320398,0.018238,0.068128,1.236109,0.0
317,NaT,NaT,0.061955,2.616436,0.164216,-0.320398,-0.811662,2.515411,0.981824,0.0
318,NaT,NaT,0.061955,-0.045120,0.164216,-0.320398,-0.185413,0.068128,0.790007,0.0


In [21]:
df.to_excel('Burari2025.xlsx', index=False)